In [0]:
%sql
-- Creating a catalog and schema 

Create catalog if not exists cdc_catalog;
create schema if not exists cdc_catalog.cdc_schema; 

use  cdc_catalog.cdc_schema;

In [0]:
%sql

--DROP TABLE IF EXISTS users_cdf;

CREATE TABLE IF NOT EXISTS users_cdf_v2
AS SELECT
  col1 AS userId,
  col2 AS name,
  col3 AS city,
  col4 AS operation,
  col5 AS sequenceNum
FROM (
  VALUES
  -- Initial load.
  (124, "Raul",     "Oaxaca",      "INSERT", 1),
  (123, "Isabel",   "Monterrey",   "INSERT", 1),
  -- New users.
  (125, "Mercedes", "Tijuana",     "INSERT", 2),
  (126, "Lily",     "Cancun",      "INSERT", 2),
  -- Isabel is removed from the system and Mercedes moved to Guadalajara.
  (123, null,       null,          "DELETE", 6),
  (125, "Mercedes", "Guadalajara", "UPDATE", 6),
  -- This batch of updates arrived out of order. The batch at sequenceNum 6 is the final state.
  (125, "Mercedes", "Mexicali",    "UPDATE", 5),
  (123, "Isabel",   "Chihuahua",   "UPDATE", 5)
  -- Uncomment to test TRUNCATE.
 --  ,(null, null,      null,          "TRUNCATE", 3)
);

In [0]:
spark.sql("select * from users_cdf").display()

In [0]:
spark.sql("select * from workspace.default.users_history").display()

In [0]:
%sql
insert into cdc_catalog.cdc_schema.users_cdf values (127, "Gunda",     "India",      "INSERT", 7)

In [0]:
%sql
insert into cdc_catalog.cdc_schema.users_cdf values (127, "Gundappa",     "India",      "UPDATE", 8)

In [0]:
spark.sql("select * from workspace.default.users_current").display()